In [14]:
# 코드 13-11 필요한 라이브러리 호출
import datetime
import os
from tensorboardX import SummaryWriter

import torch
import torchvision
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pylab as plt

import torchvision.datasets as datasets
import torchvision.transforms as transforms

# MacBook Air(Apple Silicon) GPU인 MPS 사용, 없으면 CPU
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

In [15]:
# 코드 13-12 데이터셋을 내려받은 후 텐서 변환
transform = transforms.Compose([transforms.ToTensor()])  # 픽셀값을 0~1 범위 텐서로 변환 (BCE 손실과 맞추기 위함)

train_dataset = datasets.MNIST(
    root="../chap13/data", train=True, transform=transform, download=True)

test_dataset = datasets.MNIST(
    root="../chap13/data", train=False, transform=transform, download=True)

train_loader = DataLoader(
    train_dataset, batch_size=100, shuffle=True, num_workers=4, pin_memory=False)  # pin_memory는 CUDA 전용이라 MPS에서는 False 유지

test_loader = DataLoader(
    test_dataset, batch_size=100, shuffle=False, num_workers=4)

In [16]:
# 코드 13-13 인코더 네트워크 생성
class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim):
        super(Encoder, self).__init__()
        self.input1 = nn.Linear(input_dim, hidden_dim)   # 입력(784) → 은닉층
        self.input2 = nn.Linear(hidden_dim, hidden_dim)  # 은닉층 → 은닉층
        self.mean = nn.Linear(hidden_dim, latent_dim)    # 잠재 분포의 평균 μ 출력
        self.var = nn.Linear(hidden_dim, latent_dim)     # 잠재 분포의 로그 분산 log σ² 출력 (분산이 음수가 되지 않도록 log로 학습)

        self.LeakyReLU = nn.LeakyReLU(0.2)
        self.training = True

    def forward(self, x):
        h_ = self.LeakyReLU(self.input1(x))
        h_ = self.LeakyReLU(self.input2(h_))
        mean = self.mean(h_)
        log_var = self.var(h_)
        return mean, log_var  # 인코더 네트워크에서 평균과 분산을 반환

In [17]:
# 코드 13-14 디코더 네트워크
class Decoder(nn.Module):
    def __init__(self, latent_dim, hidden_dim, output_dim):
        super(Decoder, self).__init__()
        self.hidden1 = nn.Linear(latent_dim, hidden_dim)  # 잠재 벡터 z → 은닉층
        self.hidden2 = nn.Linear(hidden_dim, hidden_dim)
        self.output = nn.Linear(hidden_dim, output_dim)   # 은닉층 → 원본 이미지 크기(784)로 복원
        self.LeakyReLU = nn.LeakyReLU(0.2)

    def forward(self, x):
        h = self.LeakyReLU(self.hidden1(x))
        h = self.LeakyReLU(self.hidden2(h))
        x_hat = torch.sigmoid(self.output(h))  # 픽셀값 범위(0~1)에 맞추기 위해 시그모이드 적용
        return x_hat  # 디코더 결과는 시그모이드를 통과했으므로 0~1 값을 갖습니다.

In [18]:
# 코드 13-15 변형 오토인코더 네트워크
class Model(nn.Module):
    def __init__(self, Encoder, Decoder):
        super(Model, self).__init__()
        self.Encoder = Encoder
        self.Decoder = Decoder

    # ① 재매개변수화 트릭: 샘플링을 미분 가능하게 만들기 위해
    #    z ~ N(μ, σ²) 대신 z = μ + σ·ε, ε ~ N(0, 1) 형태로 계산
    #    (인자 이름은 var지만 실제로 들어오는 값은 표준편차 σ)
    def reparameterization(self, mean, var):
        epsilon = torch.randn_like(var).to(device)  # randn_like는 var와 같은 디바이스(MPS)에 생성되므로 .to(device)는 사실상 생략 가능
        z = mean + var * epsilon  # z 값 구하기
        return z

    def forward(self, x):
        mean, log_var = self.Encoder(x)  # ② 인코더로 μ, log σ² 계산
        z = self.reparameterization(mean, torch.exp(0.5 * log_var))  # exp(0.5·log σ²) = σ (표준편차)
        x_hat = self.Decoder(z)  # 샘플링한 z로 이미지 복원
        return x_hat, mean, log_var  # 디코더 결과와 평균, 표준편차(log를 취한 표준편차)를 반환

In [19]:
# 코드 13-16 인코더와 디코더 객체 초기화
x_dim = 784       # 28×28 MNIST 이미지를 펼친 크기
hidden_dim = 400
latent_dim = 200  # 잠재 공간 차원
epochs = 30
batch_size = 100

encoder = Encoder(input_dim=x_dim, hidden_dim=hidden_dim, latent_dim=latent_dim)
decoder = Decoder(latent_dim=latent_dim, hidden_dim=hidden_dim, output_dim=x_dim)

model = Model(Encoder=encoder, Decoder=decoder).to(device)  # 모델 전체를 MPS로 이동

In [20]:
# 코드 13-17 손실 함수 정의
# ① VAE 손실 = 재구성 손실 + KL 발산
def loss_function(x, x_hat, mean, log_var):
    # 재구성 손실: 복원 이미지 x_hat이 원본 x와 얼마나 비슷한지 (픽셀별 BCE 합)
    reproduction_loss = nn.functional.binary_cross_entropy(x_hat, x, reduction='sum')
    # KL 발산: 인코더 분포 N(μ, σ²)가 표준정규분포 N(0, 1)에서 얼마나 벗어났는지 (정규화 역할)
    KLD = -0.5 * torch.sum(1 + log_var - mean.pow(2) - log_var.exp())
    return reproduction_loss, KLD

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [21]:
# 코드 13-18 모델 학습 함수 정의
saved_loc = 'scalar/'  # 텐서보드에서 사용할 경로
writer = SummaryWriter(saved_loc)  # ① 학습 중 손실 값을 텐서보드 로그로 기록할 객체

model.train()

def train(epoch, model, train_loader, optimizer):
    train_loss = 0
    for batch_idx, (x, _) in enumerate(train_loader):  # VAE는 비지도 학습이라 레이블(_)은 사용하지 않음
        x = x.view(batch_size, x_dim)  # (100, 1, 28, 28) → (100, 784)로 펼쳐서 Linear 층에 입력
        x = x.to(device)               # MPS로 이동

        optimizer.zero_grad()
        x_hat, mean, log_var = model(x)                   # 인코딩 → 재매개변수화 → 디코딩
        BCE, KLD = loss_function(x, x_hat, mean, log_var)  # 재구성 손실, KL 발산 계산
        loss = BCE + KLD                                  # VAE 전체 손실 (ELBO의 음수)
        writer.add_scalar("Train/Reconstruction Error", BCE.item(), batch_idx + epoch *
                          (len(train_loader.dataset)/batch_size))  # ② x축 = 전체 누적 배치 인덱스
        writer.add_scalar("Train/KL-Divergence", KLD.item(), batch_idx + epoch *
                          (len(train_loader.dataset)/batch_size))
        writer.add_scalar("Train/Total Loss", loss.item(), batch_idx + epoch *
                          (len(train_loader.dataset)/batch_size))

        train_loss += loss.item()
        loss.backward()
        optimizer.step()

        if batch_idx % 100 == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\t Loss: {:.6f}'.format(
                epoch, batch_idx * len(x), len(train_loader.dataset),
                100. * batch_idx / len(train_loader),
                loss.item() / len(x)))  # reduction='sum'이라 샘플 수로 나눠 샘플당 손실 출력

    print("======> Epoch: {} Average loss: {:.4f}".format(
        epoch, train_loss / len(train_loader.dataset)))

In [22]:
# 코드 13-19 모델 평가 함수 정의
def test(epoch, model, test_loader):
    model.eval()
    test_loss = 0
    with torch.no_grad():  # 평가 시에는 기울기 계산 생략
        for batch_idx, (x, _) in enumerate(test_loader):
            x = x.view(batch_size, x_dim)
            x = x.to(device)
            x_hat, mean, log_var = model(x)
            BCE, KLD = loss_function(x, x_hat, mean, log_var)
            loss = BCE + KLD

            writer.add_scalar("Test/Reconstruction Error", BCE.item(), batch_idx +
                              epoch * (len(test_loader.dataset)/batch_size))  # 테스트 데이터셋에 대해서도 오차를 로그에 저장
            writer.add_scalar("Test/KL-Divergence", KLD.item(), batch_idx + epoch *
                              (len(test_loader.dataset)/batch_size))
            writer.add_scalar("Test/Total Loss", loss.item(), batch_idx + epoch *
                              (len(test_loader.dataset)/batch_size))
            test_loss += loss.item()

            if batch_idx == 0:
                n = min(x.size(0), 8)  # 원본/복원 이미지 8개씩 비교
                comparison = torch.cat([x[:n], x_hat.view(batch_size, x_dim)[:n]])  # 위: 원본, 아래: 복원
                grid = torchvision.utils.make_grid(comparison.cpu())  # MPS 텐서는 이미지 저장 전 CPU로 옮겨야 함
                writer.add_image("Test image - Above: Real data, below: reconstruction data", grid, epoch)

In [23]:
# 코드 13-20 모델 학습
from tqdm.auto import tqdm
for epoch in tqdm(range(0, epochs)):
    train(epoch, model, train_loader, optimizer)
    test(epoch, model, test_loader)
    print("\n")
writer.close()  # ① 로그 파일 저장 후 writer 종료

  0%|          | 0/30 [00:00<?, ?it/s]

Train Epoch: 0 [0/60000 (0%)]	 Loss: 544.783789
Train Epoch: 0 [10000/60000 (17%)]	 Loss: 191.544883
Train Epoch: 0 [20000/60000 (33%)]	 Loss: 181.717129
Train Epoch: 0 [30000/60000 (50%)]	 Loss: 161.220566
Train Epoch: 0 [40000/60000 (67%)]	 Loss: 154.387217
Train Epoch: 0 [50000/60000 (83%)]	 Loss: 145.163027
======> Epoch: 0 Average loss: 173.7827


Train Epoch: 1 [0/60000 (0%)]	 Loss: 146.828896
Train Epoch: 1 [10000/60000 (17%)]	 Loss: 131.011436
Train Epoch: 1 [20000/60000 (33%)]	 Loss: 124.928057
Train Epoch: 1 [30000/60000 (50%)]	 Loss: 127.687119
Train Epoch: 1 [40000/60000 (67%)]	 Loss: 128.623486
Train Epoch: 1 [50000/60000 (83%)]	 Loss: 124.977881
======> Epoch: 1 Average loss: 129.0855


Train Epoch: 2 [0/60000 (0%)]	 Loss: 122.075254
Train Epoch: 2 [10000/60000 (17%)]	 Loss: 119.754531
Train Epoch: 2 [20000/60000 (33%)]	 Loss: 115.351396
Train Epoch: 2 [30000/60000 (50%)]	 Loss: 114.245469
Train Epoch: 2 [40000/60000 (67%)]	 Loss: 119.832676
Train Epoch: 2 [50000/60000 (8

In [24]:
# 코드 13-21 텐서보드에서 오차 확인
# 매직 명령어(%)는 줄 끝의 주석까지 인자로 인식하므로 주석은 별도 줄에 작성
# ① 주피터에서 텐서보드 확장 로드
%load_ext tensorboard
# ② scalar 폴더의 로그를 6013 포트로 실행
%tensorboard --logdir scalar --port=6013

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6013 (pid 26374), started 0:04:58 ago. (Use '!kill 26374' to kill it.)